In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

/home/diego/miniconda3/envs/DC/lib/python3.10/site-packages/sksurv/__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
train_folder_path = "./data/X_train/"
test_folder_path = "./data/X_test/"

# Clinical Data
df = pd.read_csv(train_folder_path + "clinical_train.csv", sep=",")
df_eval = pd.read_csv(test_folder_path + "clinical_test.csv", sep=",")

# Molecular Data
maf_df = pd.read_csv(train_folder_path + "molecular_train.csv", sep=",")
maf_eval = pd.read_csv(test_folder_path + "molecular_test.csv", sep=",")

target_df = pd.read_csv(train_folder_path + "target_train.csv", sep=",")
target_df["OS_YEARS"] = pd.to_numeric(target_df["OS_YEARS"], errors="coerce")
target_df["OS_STATUS"] = target_df["OS_STATUS"].astype(bool)
#target_df_test = pd.read_csv("./data/target_test.csv")

# Preview the data
target_df.head()

,ID,OS_YEARS,OS_STATUS
0,P132697,1.115068,True
1,P132698,4.928767,False
2,P116889,2.043836,False
3,P132699,2.476712,True
4,P132700,3.145205,False


### Step 1: Data Preparation (clinical data only)

For survival analysis, we’ll format the dataset so that OS_YEARS represents the time variable and OS_STATUS represents the event indicator.

In [4]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Select features
features = ['BM_BLAST', 'HB', 'PLT']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df.loc[df['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

OS_STATUS       bool
OS_YEARS     float64
dtype: object


### Step 2: Splitting the Dataset
We’ll split the data into training and testing sets to evaluate the model’s performance.

In [5]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [6]:
# Survival-aware imputation for missing values
imputer = SimpleImputer(strategy="median")
X_train[['BM_BLAST', 'HB', 'PLT']] = imputer.fit_transform(X_train[['BM_BLAST', 'HB', 'PLT']])
X_test[['BM_BLAST', 'HB', 'PLT']] = imputer.transform(X_test[['BM_BLAST', 'HB', 'PLT']])

### Step 3: Training Standard Machine Learning Methods

In this step, we train a standard LightGBM model on survival data, but we do not account for censoring. Instead of treating the event status, we use only the observed survival times as the target variable. This approach disregards whether an individual’s event (e.g., death) was observed or censored, effectively treating the problem as a standard regression task. While this method provides a basic benchmark, it may be less accurate than survival-specific models (but still be explored!), as it does not leverage the information contained in censored observations.

In [40]:
# Import necessary libraries
import lightgbm as lgb
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# Define LightGBM parameters
lgbm_params = {
    'max_depth': 3,
    'learning_rate': 0.05,
    'verbose': -1
}

# Prepare the data for LightGBM
# Scale the target (OS_YEARS) to reduce skew, apply weights based on event status
X_train_lgb = X_train  # Features for training
y_train_transformed = y_train['OS_YEARS']

# Create LightGBM dataset
train_dataset = lgb.Dataset(X_train_lgb, label=y_train_transformed)

# Train the LightGBM model
model = lgb.train(params=lgbm_params, train_set=train_dataset)

# Make predictions on the training and testing sets
pred_train = -model.predict(X_train)
pred_test = -model.predict(X_test)

# Evaluate the model using Concordance Index IPCW
train_ci_ipcw = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
test_ci_ipcw = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]
print(f"LightGBM Survival Model Concordance Index IPCW on train: {train_ci_ipcw:.2f}")
print(f"LightGBM Survival Model Concordance Index IPCW on test: {test_ci_ipcw:.2f}")

LightGBM Survival Model Concordance Index IPCW on train: 0.69
LightGBM Survival Model Concordance Index IPCW on test: 0.65


### Step 4: Cox Proportional Hazards Model

To account for censoring in survival analysis, we use a Cox Proportional Hazards (Cox PH) model, a widely used method that estimates the effect of covariates on survival times without assuming a specific baseline survival distribution. The Cox PH model is based on the hazard function, $h(t | X)$, which represents the instantaneous risk of an event (e.g., death) at time $t$ given covariates $X$. The model assumes that the hazard can be expressed as:

$$h(t | X) = h_0(t) \exp(\beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p)$$


where $h_0(t)$ is the baseline hazard function, and $\beta$ values are coefficients for each covariate, representing the effect of $X$ on the hazard. Importantly, the proportional hazards assumption implies that the hazard ratios between individuals are constant over time. This approach effectively leverages both observed and censored survival times, making it a more suitable method for survival data compared to standard regression techniques that ignore censoring.

In [41]:
# Initialize and train the Cox Proportional Hazards model
cox = CoxPHSurvivalAnalysis()
cox.fit(X_train, y_train)

# Evaluate the model using Concordance Index IPCW
cox_cindex_train = concordance_index_ipcw(y_train, y_train, cox.predict(X_train), tau=7)[0]
cox_cindex_test = concordance_index_ipcw(y_train, y_test, cox.predict(X_test), tau=7)[0]
print(f"Cox Proportional Hazard Model Concordance Index IPCW on train: {cox_cindex_train:.2f}")
print(f"Cox Proportional Hazard Model Concordance Index IPCW on test: {cox_cindex_test:.2f}")

Cox Proportional Hazard Model Concordance Index IPCW on train: 0.66
Cox Proportional Hazard Model Concordance Index IPCW on test: 0.66


### Step 5: Naive Approach to Incorporate Mutations

In this step, we take a very naive approach to account for genetic mutations by simply counting the total number of somatic mutations per patient. Instead of analyzing specific mutations or their biological impact, we use this aggregate count as a basic feature to reflect the mutational burden for each individual. Although simplistic, this feature can serve as a general indicator of genetic variability across patients, which may influence survival outcomes. More sophisticated mutation analysis could be incorporated in future models to improve predictive power.

In [9]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_2 = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})

In [10]:
# Select features
features = ['BM_BLAST', 'HB', 'PLT', 'Nmut']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df_2.loc[df_2['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

In [11]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [12]:
# Survival-aware imputation for missing values
imputer = SimpleImputer(strategy="median")
X_train[['BM_BLAST', 'HB', 'PLT', 'Nmut']] = imputer.fit_transform(X_train[['BM_BLAST', 'HB', 'PLT', 'Nmut']])
X_test[['BM_BLAST', 'HB', 'PLT', 'Nmut']] = imputer.transform(X_test[['BM_BLAST', 'HB', 'PLT', 'Nmut']])

In [13]:
# Initialize and train the Cox Proportional Hazards model
cox = CoxPHSurvivalAnalysis()
cox.fit(X_train, y_train)

# Evaluate the model using Concordance Index IPCW
cox_cindex_train = concordance_index_ipcw(y_train, y_train, cox.predict(X_train), tau=7)[0]
cox_cindex_test = concordance_index_ipcw(y_train, y_test, cox.predict(X_test), tau=7)[0]
print(f"Cox Proportional Hazard Model Concordance Index IPCW on train: {cox_cindex_train:.2f}")
print(f"Cox Proportional Hazard Model Concordance Index IPCW on test: {cox_cindex_test:.2f}")

Cox Proportional Hazard Model Concordance Index IPCW on train: 0.68
Cox Proportional Hazard Model Concordance Index IPCW on test: 0.68


### Step 6: Find other features to improve the prediction

Now we will try searching the best features to put in our predictors

In [7]:
# Compute max depth across all patients per mutation identity
maf_df["MAX_DEPTH_GLOBAL"] = (
    maf_df.groupby(["CHR", "START", "REF", "ALT"])["DEPTH"].transform("max")
)

# You now have both:
# - DEPTH: the depth in this specific patient
# - MAX_DEPTH_GLOBAL: the maximum depth seen in any patient for this same mutation
maf_df[["ID", "GENE", "DEPTH", "MAX_DEPTH_GLOBAL"]].head(10)

,ID,GENE,DEPTH,MAX_DEPTH_GLOBAL
0,P100000,CBL,1308.0,1308.0
1,P100000,IRF1,532.0,532.0
2,P100000,ROBO2,876.0,876.0
3,P100000,TET2,826.0,826.0
4,P100000,DNMT3A,942.0,942.0
5,P100001,CHEK2,514.0,514.0
6,P100001,PIK3CA,558.0,1083.0
7,P100002,PIK3CA,310.0,1083.0
8,P100002,TP53,487.0,487.0
9,P100004,STAG2,1296.0,1296.0


### Compute Weighted Mutation Burden per Patient

In this step, we create a new feature called **`Nmut_2`**, representing a *weighted mutation burden* for each patient.  
Instead of simply counting the number of mutations, we account for both the **variant allele frequency (VAF)** and the **sequencing depth**, normalized by the global maximum depth across all samples.

#### Formula

For each mutation *i* in patient *j*:

$$
\text{weighted\_effect}_{i,j} \;=\; \frac{\text{VAF}_{i,j} \times \text{DEPTH}_{i,j}}{\text{MAX\_DEPTH\_GLOBAL}}
$$

**Per-patient aggregated score (displayed):**

$$
\text{Nmut\_2}_j \;=\; \sum_{i \in \text{mutations of patient } j} \text{weighted\_effect}_{i,j}
$$


In [59]:
# Step 1: Compute weighted mutation burden per patient
tmp = (
    maf_df
    .assign(weighted_effect = maf_df["VAF"] * maf_df["DEPTH"] / maf_df["MAX_DEPTH_GLOBAL"])
    .groupby("ID", as_index=False)["weighted_effect"]
    .sum()
    .rename(columns={"weighted_effect": "Nmut_2"})
)

# Step 2: Merge with the clinical dataset
df_3 = (
    df_2
    .merge(tmp, on="ID", how="left")
    .fillna({"Nmut_2": 0})  # patients with no mutations get Nmut = 0
)

df_3.head()


,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,Nmut,Nmut_2
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",9.0,2.124783
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx",3.0,0.716630
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",3.0,0.102628
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",11.0,2.106211
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1.0,0.268213


In [70]:
from itertools import combinations
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

results = []

# Merge with the clinical and molecular training dataset 
all_features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'ANC', 'MONOCYTES', 'Nmut', 'Nmut_2']
base_model = CoxnetSurvivalAnalysis()

for k in range(1, 9):
    for subset in combinations(all_features, k):
        subset = list(subset)

        # Select features and target
        X = df_3.loc[df_3['ID'].isin(target_df["ID"]), subset].copy()
        y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

        # Handle missing values
        imputer = SimpleImputer(strategy="median")
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=subset)

        # Train/Test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_imputed, y, test_size=0.3, random_state=42
        )

        # Fit Cox model
        model = base_model.fit(X_train, y_train)

        # Predict risk scores
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        # Compute Concordance Index IPCW
        train_ci = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
        test_ci = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]

        # Store results
        results.append({
            "Features": subset,
            "Train C-index": train_ci,
            "Test C-index": test_ci
        })

# ===============================================================
# Display Results
# ===============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Test C-index", ascending=False)
    .reset_index(drop=True)
)

print("✅ Top-performing feature subsets:")
display(results_df.head(10))

✅ Top-performing feature subsets:


,Features,Train C-index,Test C-index
0,"[BM_BLAST, HB, PLT, WBC, MONOCYTES, Nmut]",0.685726,0.685607
1,"[BM_BLAST, HB, PLT, WBC, ANC, MONOCYTES, Nmut]",0.685784,0.685560
2,"[BM_BLAST, HB, PLT, WBC, Nmut]",0.685677,0.685363
3,"[BM_BLAST, HB, PLT, WBC, ANC, Nmut]",0.685745,0.685324
4,"[BM_BLAST, HB, PLT, ANC, MONOCYTES, Nmut]",0.686327,0.685300
5,"[BM_BLAST, HB, PLT, ANC, Nmut]",0.686334,0.685294
6,"[BM_BLAST, HB, PLT, MONOCYTES, Nmut]",0.684962,0.683756
7,"[BM_BLAST, HB, PLT, Nmut]",0.684431,0.683280
8,"[BM_BLAST, HB, PLT, WBC, MONOCYTES, Nmut, Nmut_2]",0.689440,0.683208
9,"[BM_BLAST, HB, PLT, WBC, ANC, MONOCYTES, Nmut,...",0.689317,0.683121


### Conclusion
Nmut appears to work better than Nmut2

In [60]:
# Step 1: Compute weighted mutation burden per patient
tmp = (
    maf_df
    .assign(VAF_effect = maf_df["VAF"])
    .groupby("ID", as_index=False)["VAF_effect"]
    .sum()
    .rename(columns={"VAF_effect": "Nmut_3"})
)

# Step 2: Merge with the clinical dataset
df_4 = (
    df_2
    .merge(tmp, on="ID", how="left")
    .fillna({"Nmut_3": 0})  # patients with no mutations get Nmut = 0
)

df_4.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,Nmut,Nmut_3
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",9.0,2.2642
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx",3.0,0.8186
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",3.0,0.1180
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",11.0,2.3015
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1.0,0.4721


In [61]:
from itertools import combinations
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

results = []

# Merge with the clinical and molecular training dataset 
all_features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'ANC', 'MONOCYTES', 'Nmut', 'Nmut_3']
base_model = CoxnetSurvivalAnalysis()

for k in range(1, 9):
    for subset in combinations(all_features, k):
        subset = list(subset)

        # Select features and target
        X = df_4.loc[df_4['ID'].isin(target_df["ID"]), subset].copy()
        y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

        # Handle missing values
        imputer = SimpleImputer(strategy="median")
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=subset)

        # Train/Test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_imputed, y, test_size=0.3, random_state=42
        )

        # Fit Cox model
        model = base_model.fit(X_train, y_train)

        # Predict risk scores
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        # Compute Concordance Index IPCW
        train_ci = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
        test_ci = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]

        # Store results
        results.append({
            "Features": subset,
            "Train C-index": train_ci,
            "Test C-index": test_ci
        })

# ===============================================================
# Display Results
# ===============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Test C-index", ascending=False)
    .reset_index(drop=True)
)

print("✅ Top-performing feature subsets:")
display(results_df.head(10))

✅ Top-performing feature subsets:


,Features,Train C-index,Test C-index
0,"[BM_BLAST, HB, PLT, WBC, MONOCYTES, Nmut]",0.685726,0.685607
1,"[BM_BLAST, HB, PLT, WBC, ANC, MONOCYTES, Nmut]",0.685784,0.685560
2,"[BM_BLAST, HB, PLT, WBC, Nmut]",0.685677,0.685363
3,"[BM_BLAST, HB, PLT, WBC, ANC, Nmut]",0.685745,0.685324
4,"[BM_BLAST, HB, PLT, ANC, MONOCYTES, Nmut]",0.686327,0.685300
5,"[BM_BLAST, HB, PLT, ANC, Nmut]",0.686334,0.685294
6,"[BM_BLAST, HB, PLT, MONOCYTES, Nmut]",0.684962,0.683756
7,"[BM_BLAST, HB, PLT, Nmut]",0.684431,0.683280
8,"[BM_BLAST, HB, PLT, WBC, ANC, MONOCYTES, Nmut,...",0.689877,0.682413
9,"[BM_BLAST, HB, PLT, WBC, Nmut, Nmut_3]",0.690005,0.682384


### Gene Mutations

We aimed to identify the mutated genes that most significantly influence our survival predictions.
To achieve this, we quantified the mutational impact of each gene by assigning, for every patient, the maximum Variant Allele Frequency (VAF) observed across all mutations occurring within that gene.
If a given gene was not mutated in a patient, a value of zero was assigned to ensure consistent representation across all individuals.

In [54]:
# Make a copy of your mutation dataframe
maf_df_copy = maf_df.copy()

# Ensure proper datatypes
maf_df_copy["VAF"] = pd.to_numeric(maf_df_copy["VAF"], errors="coerce")
maf_df_copy["ID"] = maf_df_copy["ID"].astype(str)
maf_df_copy["GENE"] = maf_df_copy["GENE"].astype(str)

# Step 1️⃣: Pivot to create a matrix of patients (rows) × genes (columns)
# If a patient has multiple mutations in the same gene, take the max VAF
gene_matrix = (
    maf_df_copy
    .groupby(["ID", "GENE"])["VAF"]
    .max()                          # use max VAF per gene per patient
    .unstack(fill_value=0)          # convert to wide format
    .reset_index()
)

# Step 2️⃣: Merge with your clinical dataframe
df_with_genes = df_2.merge(gene_matrix, on="ID", how="left")

# Step 3️⃣: Replace missing values (patients without a given gene mutation)
df_with_genes = df_with_genes.fillna(0)

# ✅ Now df_with_genes has one column per gene (e.g. TP53, TET2, DNMT3A, etc.)
# Each value = VAF if mutated, or 0 if not mutated
print(df_with_genes.shape)
#df_with_genes.head()
df_with_genes.iloc[:, 9:].head()

(3323, 134)


,Nmut,ABL1,ARID1A,ARID2,ASXL1,ASXL2,ATRX,BAP1,BCL10,BCOR,...,TET2,TP53,U2AF1,U2AF2,WHSC1,WT1,ZBTB33,ZMYM3,ZNF318,ZRSR2
0,9.0,0.0,0.0,0.0000,0.3553,0.0,0.0,0.0,0.0,0.0,...,0.394,0.0,0.35,0.0,0.0,0.000,0.0,0.0,0.0,0.000
1,3.0,0.0,0.0,0.0000,0.2825,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.000
2,3.0,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.035,0.0,0.0,0.0,0.000
3,11.0,0.0,0.0,0.0984,0.2871,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.049
4,1.0,0.0,0.0,0.0000,0.4721,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.000


In [55]:
from itertools import combinations
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

results = []

# Merge with the clinical and molecular training dataset 
fixed_features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut','TP53','TET2','SF3B1','CBL']
features_choose = df_with_genes.columns[10:].tolist()
base_model = CoxnetSurvivalAnalysis()

for k in range(1, 2):
    for subset in combinations(features_choose, k):
        subset = list(subset)
        selected_features = list(set(fixed_features + subset))

        # Select features and target
        X = df_with_genes.loc[df_with_genes['ID'].isin(target_df["ID"]), selected_features].copy()
        y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

        # Handle missing values
        imputer = SimpleImputer(strategy="median")
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=selected_features)

        # Train/Test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_imputed, y, test_size=0.3, random_state=42
        )

        # Fit Cox model
        model = base_model.fit(X_train, y_train)

        # Predict risk scores
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        # Compute Concordance Index IPCW
        train_ci = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
        test_ci = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]

        # Store results
        results.append({
            "Features": subset,
            "Train C-index": train_ci,
            "Test C-index": test_ci
        })

# ===============================================================
# Display Results
# ===============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Test C-index", ascending=False)
    .reset_index(drop=True)
)

print("✅ Top-performing feature subsets:")
display(results_df.head(10))

✅ Top-performing feature subsets:


,Features,Train C-index,Test C-index
0,[ABL1],0.699809,0.704106
1,[KMT2D],0.699809,0.704106
2,[RAD50],0.699809,0.704106
3,[RAD21],0.699809,0.704106
4,[RAC1],0.699809,0.704106
5,[PTPRF],0.699809,0.704106
6,[PTPN11],0.699809,0.704106
7,[PTEN],0.699809,0.704106
8,[PRPF8],0.699809,0.704106
9,[PRPF40A],0.699809,0.704106


In [64]:
# Make a copy of your mutation dataframe
maf_df_copy = maf_df.copy()

# Ensure proper datatypes
maf_df_copy["VAF"] = pd.to_numeric(maf_df_copy["VAF"], errors="coerce")
maf_df_copy["ID"] = maf_df_copy["ID"].astype(str)
maf_df_copy["GENE"] = maf_df_copy["GENE"].astype(str)

# Step 1️⃣: Pivot to create a matrix of patients (rows) × genes (columns)
# If a patient has multiple mutations in the same gene, take the max VAF
gene_matrix = (
    maf_df_copy
    .groupby(["ID", "GENE"])["VAF"]
    .sum()                          # sum all VAF per gene per patient
    .unstack(fill_value=0)          # convert to wide format
    .reset_index()
)

# Step 2️⃣: Merge with your clinical dataframe
df_with_genes_sum = df_2.merge(gene_matrix, on="ID", how="left")

# Step 3️⃣: Replace missing values (patients without a given gene mutation)
df_with_genes_sum = df_with_genes_sum.fillna(0)

# ✅ Now df_with_genes has one column per gene (e.g. TP53, TET2, DNMT3A, etc.)
# Each value = VAF if mutated, or 0 if not mutated
print(df_with_genes_sum.shape)
#df_with_genes.head()
df_with_genes_sum.iloc[:, 9:].head()

(3323, 134)


,Nmut,ABL1,ARID1A,ARID2,ASXL1,ASXL2,ATRX,BAP1,BCL10,BCOR,...,TET2,TP53,U2AF1,U2AF2,WHSC1,WT1,ZBTB33,ZMYM3,ZNF318,ZRSR2
0,9.0,0.0,0.0,0.0000,0.3553,0.0,0.0,0.0,0.0,0.0,...,0.394,0.0,0.35,0.0,0.0,0.000,0.0,0.0,0.0,0.000
1,3.0,0.0,0.0,0.0000,0.2825,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.000
2,3.0,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.035,0.0,0.0,0.0,0.000
3,11.0,0.0,0.0,0.0984,0.2871,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.049
4,1.0,0.0,0.0,0.0000,0.4721,0.0,0.0,0.0,0.0,0.0,...,0.000,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.000


In [ ]:
from itertools import combinations
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

results = []

# Merge with the clinical and molecular training dataset 
fixed_features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut','TP53',
                  'TET2','SF3B1','CBL','ZRSR2','EZH2','U2AF1']
features_choose = df_with_genes_sum.columns[10:].tolist()
base_model = CoxnetSurvivalAnalysis()

for k in range(1, 2):
    for subset in combinations(features_choose, k):
        subset = list(subset)
        selected_features = list(set(fixed_features + subset))

        # Select features and target
        X = df_with_genes_sum.loc[df_with_genes_sum['ID'].isin(target_df["ID"]), selected_features].copy()
        y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

        # Handle missing values
        imputer = SimpleImputer(strategy="median")
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=selected_features)

        # Train/Test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_imputed, y, test_size=0.3, random_state=42
        )

        # Fit Cox model
        model = base_model.fit(X_train, y_train)

        # Predict risk scores
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        # Compute Concordance Index IPCW
        train_ci = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
        test_ci = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]

        # Store results
        results.append({
            "Features": subset,
            "Train C-index": train_ci,
            "Test C-index": test_ci
        })

# ===============================================================
# Display Results
# ===============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Test C-index", ascending=False)
    .reset_index(drop=True)
)

print("✅ Top-performing feature subsets:")
display(results_df.head(10))

✅ Top-performing feature subsets:


,Features,Train C-index,Test C-index
0,[U2AF1],0.703678,0.706912
1,[ABL1],0.703652,0.706867
2,[NXF1],0.703652,0.706867
3,[RAD21],0.703652,0.706867
4,[RAC1],0.703652,0.706867
5,[PTPRF],0.703652,0.706867
6,[PTPN11],0.703652,0.706867
7,[PTEN],0.703652,0.706867
8,[PRPF8],0.703652,0.706867
9,[PRPF40A],0.703652,0.706867


In [53]:
from sklearn import model_selection
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from time import time

#features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut']
#X = df_3.loc[df_3['ID'].isin(target_df["ID"]), features].copy()
features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut','TP53','TET2','SF3B1','CBL']
X = df_with_genes.loc[df_with_genes['ID'].isin(target_df["ID"]), features].copy()
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=features)
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.3, random_state=42
)

start = time()

my_kfold = KFold(n_splits=5, shuffle=True, random_state=0)

tuned_parameters = {'max_depth': [5], 
                    'min_samples_split': [2],
                    'n_estimators': [300],
                    'learning_rate':[0.05]
                    }

GBM = GridSearchCV(GradientBoostingSurvivalAnalysis(),
                      tuned_parameters,
                      cv=my_kfold)

model = GBM.fit(X_train, y_train)

# Predict risk scores
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

# Compute Concordance Index IPCW
train_ci_ipcw = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
test_ci_ipcw = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]

print(f"GBM Survival Model Concordance Index IPCW on train: {train_ci_ipcw:.2f}")
print(f"GBM Survival Model Concordance Index IPCW on test: {test_ci_ipcw:.2f}")



GBM Survival Model Concordance Index IPCW on train: 0.83
GBM Survival Model Concordance Index IPCW on test: 0.69


In [51]:
print("✅ Best Parameters:", GBM.best_params_)

✅ Best Parameters: {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 300}


In [62]:
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

#features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut']
#X = df_3.loc[df_3['ID'].isin(target_df["ID"]), features].copy()
features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut','TP53','TET2','SF3B1','CBL']
X = df_with_genes.loc[df_with_genes['ID'].isin(target_df["ID"]), features].copy()
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=features)
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.3, random_state=42
)

# Prepare the data for LightGBM
# Scale the target (OS_YEARS) to reduce skew, apply weights based on event status
X_train_XGBR = X_train  # Features for training
y_train_transformed = y_train['OS_YEARS']

my_kfold = KFold(n_splits=5, shuffle=True, random_state=0)

tuned_parameters = {"n_estimators": [100, 300, 500, 1000],
                    "learning_rate": [0.01, 0.05],
                    "reg_alpha": [0, 0.001, 0.1, 1, 10],
                    "reg_lambda": [0, 0.001, 0.1, 1, 10],
                    "max_depth": [3, 5, 8]
                    }

# tuned_parameters = {"n_estimators": [100],
#                     "learning_rate": [0.05],
#                     "reg_alpha": [0.001],
#                     "reg_lambda": [10],
#                     "max_depth": [3]
#                     }

XGBR = GridSearchCV(XGBRegressor(),
                      tuned_parameters,
                      cv=my_kfold)

# Train the LightGBM model
model = XGBR.fit(X_train_XGBR, y_train_transformed)

# Make predictions on the training and testing sets
pred_train = -model.predict(X_train_XGBR)
pred_test = -model.predict(X_test)

# Evaluate the model using Concordance Index IPCW
train_ci_ipcw = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
test_ci_ipcw = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]
print(f"XGBRegressor Model Concordance Index IPCW on train: {train_ci_ipcw:.2f}")
print(f"XGBRegressor Model Concordance Index IPCW on test: {test_ci_ipcw:.2f}")

XGBRegressor Model Concordance Index IPCW on train: 0.75
XGBRegressor Model Concordance Index IPCW on test: 0.69


In [63]:
print("✅ Best Parameters:", XGBR.best_params_)

✅ Best Parameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_alpha': 10, 'reg_lambda': 1}


In [47]:
# Ensure VAF is numeric
maf_df["VAF"] = pd.to_numeric(maf_df["VAF"], errors="coerce").fillna(0)

# Create a unique mutation identifier (e.g., TP53_p.R248Q)
maf_df["GENE_MUT"] = maf_df["GENE"].astype(str) + "_" + maf_df["PROTEIN_CHANGE"].astype(str)

# Pivot the table: one column per unique mutation, value = VAF
mutation_vaf = (
    maf_df
    .groupby(["ID", "GENE_MUT"])["VAF"]
    .max()  # in case multiple records for same mutation per patient
    .unstack(fill_value=0)  # turn into wide format
    .reset_index()
)

# Merge with your clinical dataset
df_with_mutations = (
    df_2
    .merge(mutation_vaf, on="ID", how="left")
    .fillna(0)  # if patient has no mutation, fill with 0s
)

df_with_mutations.head()


,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,Nmut,...,ZRSR2_p.W153*,ZRSR2_p.W291*,ZRSR2_p.W307*,ZRSR2_p.W340*,ZRSR2_p.W75*,ZRSR2_p.Y175C,ZRSR2_p.Y240*,ZRSR2_p.Y347*,ZRSR2_p.Y373fs*12,ZRSR2_p.Y374*
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx",3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
from itertools import combinations
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

results = []

# Merge with the clinical and molecular training dataset 
fixed_features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut']
features_choose = df_with_mutations.columns[10:].tolist()
base_model = CoxnetSurvivalAnalysis()

for k in range(1, 2):
    for subset in combinations(features_choose, k):
        subset = list(subset)
        selected_features = list(set(fixed_features + subset))

        # Select features and target
        X = df_with_mutations.loc[df_with_mutations['ID'].isin(target_df["ID"]), selected_features].copy()
        y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

        # Handle missing values
        imputer = SimpleImputer(strategy="median")
        X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=selected_features)

        # Train/Test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_imputed, y, test_size=0.3, random_state=42
        )

        # Fit Cox model
        model = base_model.fit(X_train, y_train)

        # Predict risk scores
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        # Compute Concordance Index IPCW
        train_ci = concordance_index_ipcw(y_train, y_train, pred_train, tau=7)[0]
        test_ci = concordance_index_ipcw(y_train, y_test, pred_test, tau=7)[0]

        # Store results
        results.append({
            "Features": subset,
            "Train C-index": train_ci,
            "Test C-index": test_ci
        })

# ===============================================================
# Display Results
# ===============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Test C-index", ascending=False)
    .reset_index(drop=True)
)

print("✅ Top-performing feature subsets:")
display(results_df.head(10))

✅ Top-performing feature subsets:


,Features,Train C-index,Test C-index
0,[ABL1_p.K253Q],0.673476,0.679878
1,[STAG2_p.S1075*],0.673476,0.679878
2,[STAG2_p.R807_D808ins*],0.673476,0.679878
3,[STAG2_p.R766fs*18],0.673476,0.679878
4,[STAG2_p.R667fs*1],0.673476,0.679878
5,[STAG2_p.R667W],0.673476,0.679878
6,[STAG2_p.R614*],0.673476,0.679878
7,[STAG2_p.R604Q],0.673476,0.679878
8,[STAG2_p.R604*],0.673476,0.679878
9,[STAG2_p.R450fs*20],0.673476,0.679878
